In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                       "huggingface_hub<1.0", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
DATASET = "pathmnist"  # pathmnist | histoset | skintissue
SEED = 42  # any int; one seed per run
FEATURE_DIR = "/kaggle/working/features"

PARALLEL = True  # True | False

MMAP_CACHE_DIR = "/kaggle/working/npz_mmap"

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading facebook/dinov2-base ...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import yaml
import torch

from data.loaders import get_data_loaders, get_sample_ids
from data.identity import sample_order_fingerprint
from data.npz_mmap import export_npz_to_npy
from features.visual import (
    _feature_cache_paths,
    assemble_feature_shards,
    get_or_extract_features,
)
from scripts.extract_visual_features import build_shard_jobs, extract_shard_on_worker
from utils import visual_archive_stem
from utils.parallel import run_variants_parallel, visible_gpu_count

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

VIT_NAME = config.get("models", {}).get("vit", "facebook/dinov2-base")
SAFE_VIT = VIT_NAME.replace("/", "_")

assert torch.cuda.is_available(), "Attach a Kaggle GPU before extraction"
assert Path(DATA_PATHS[DATASET]).exists(), f"Missing Kaggle input: {DATA_PATHS[DATASET]}"
assert not str(FEATURE_DIR).startswith("/kaggle/input"), (
    "FEATURE_DIR must be writable; /kaggle/input is read-only"
)
Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)

SHARDS = visible_gpu_count() if PARALLEL else 1
SHARDS = max(1, SHARDS)

import shutil as _shutil

total_npz = (
    Path(DATA_PATHS[DATASET]).stat().st_size
    if DATA_PATHS[DATASET].endswith(".npz") else 0
)
free_disk = _shutil.disk_usage("/kaggle/working").free if Path("/kaggle/working").exists() \
    else _shutil.disk_usage(".").free
print(f"backbone={VIT_NAME} cache={FEATURE_DIR}")
print(f"GPUs visible: {visible_gpu_count()} | shards per job: {SHARDS}")
if total_npz:
    print(f"npz inputs: {total_npz / 2**30:.1f} GiB | mmap export needs about the same")
    print(f"free disk : {free_disk / 2**30:.1f} GiB at {MMAP_CACHE_DIR}")
    assert free_disk > total_npz * 1.1, (
        f"Not enough scratch disk for the .npy export: need ~{total_npz / 2**30:.1f} GiB, "
        f"have {free_disk / 2**30:.1f} GiB. Extract one dataset at a time."
    )
try:
    with open("/proc/meminfo") as handle:
        mem_total = int(next(l for l in handle if l.startswith("MemTotal")).split()[1]) * 1024
    print(f"system RAM: {mem_total / 2**30:.1f} GiB across {SHARDS} worker(s)")
except (OSError, StopIteration, ValueError):
    pass

In [ ]:
import time

started = time.time()
print("=" * 70)
print(f"{DATASET} | seed {SEED} | {VIT_NAME}")
train_path, test_path, manifest_path = _feature_cache_paths(
    FEATURE_DIR, DATASET, SEED, VIT_NAME
)

if Path(manifest_path).is_file():
    print("  complete cache exists, skipping:", Path(manifest_path).name)
else:
    mmap_dir = None
    if DATA_PATHS[DATASET].endswith(".npz"):
        export_npz_to_npy(DATA_PATHS[DATASET], MMAP_CACHE_DIR)
        mmap_dir = MMAP_CACHE_DIR

    train_loader, test_loader, _ = get_data_loaders(
        DATA_PATHS[DATASET], SEED, verbose=True, mmap_cache_dir=mmap_dir
    )
    n_train, n_test = len(train_loader.dataset), len(test_loader.dataset)
    train_fingerprint = sample_order_fingerprint(get_sample_ids(train_loader.dataset))
    test_fingerprint = sample_order_fingerprint(get_sample_ids(test_loader.dataset))
    print(f"  mmap={getattr(train_loader.dataset, 'mmap', None)}")
    del train_loader, test_loader

    if SHARDS == 1:
        train_loader, test_loader, _ = get_data_loaders(
            DATA_PATHS[DATASET], SEED, mmap_cache_dir=mmap_dir
        )
        get_or_extract_features(
            train_loader, test_loader, DATASET, SEED, VIT_NAME,
            torch.device("cuda:0"), cache_dir=FEATURE_DIR,
            train_fingerprint=train_fingerprint,
            test_fingerprint=test_fingerprint,
        )
        del train_loader, test_loader
    else:
        jobs = build_shard_jobs(
            DATA_PATHS[DATASET], DATASET, SEED, VIT_NAME, SHARDS, FEATURE_DIR,
            mmap_cache_dir=mmap_dir,
        )
        results = run_variants_parallel(
            jobs, extract_shard_on_worker, num_workers=SHARDS
        )
        failed = [r["label"] for r in results if not r["ok"]]
        assert not failed, f"shards failed: {failed}"
        train_features, test_features = assemble_feature_shards(
            DATASET, SEED, VIT_NAME, SHARDS, n_train, n_test,
            cache_dir=FEATURE_DIR,
            train_fingerprint=train_fingerprint,
            test_fingerprint=test_fingerprint,
        )
        print(f"  train {train_features.shape} | test {test_features.shape}")
        del train_features, test_features

print("=" * 70)
print(f"total {time.time() - started:.0f}s")

In [ ]:
import json
import shutil

import numpy as np

train_path, test_path, manifest_path = _feature_cache_paths(
    FEATURE_DIR, DATASET, SEED, VIT_NAME
)
manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))
train = np.load(train_path, mmap_mode="r")
test = np.load(test_path, mmap_mode="r")
assert manifest["dataset"] == DATASET and manifest["seed"] == SEED
assert manifest["backbone"] == VIT_NAME
assert list(train.shape) == manifest["train_shape"]
assert list(test.shape) == manifest["test_shape"]
assert np.all(np.isfinite(train[:256])) and np.all(np.isfinite(test[:256]))
print(f"OK {DATASET} seed{SEED}: train{tuple(train.shape)} test{tuple(test.shape)}")

leftover = list(Path(FEATURE_DIR).glob(".shards_*"))
assert not leftover, f"unassembled shard directories remain: {leftover}"

if MMAP_CACHE_DIR and Path(MMAP_CACHE_DIR).is_dir():
    freed = sum(f.stat().st_size for f in Path(MMAP_CACHE_DIR).rglob("*") if f.is_file())
    shutil.rmtree(MMAP_CACHE_DIR, ignore_errors=True)
    print(f"removed mmap export, freed {freed / 2**30:.1f} GiB")

In [ ]:
import json
import shutil

SOURCE = Path(FEATURE_DIR)
WORKING = Path("/kaggle/working")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
assert SOURCE.resolve() != WORKING.resolve(), (
    "FEATURE_DIR must be a subdirectory of /kaggle/working, not /kaggle/working itself"
)

STEM = visual_archive_stem(DATASET, SEED, VIT_NAME)
ARCHIVE = WORKING / STEM
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6

print(f"{ARCHIVE.name}.zip  ({size_mb:.1f} MB) contains:")
for path in sorted(SOURCE.iterdir()):
    print(f"    {path.name}  ({path.stat().st_size / 1e6:.1f} MB)")

shutil.rmtree(SOURCE, ignore_errors=True)
if MMAP_CACHE_DIR and Path(MMAP_CACHE_DIR).is_dir():
    shutil.rmtree(MMAP_CACHE_DIR, ignore_errors=True)

remaining = sorted(p for p in WORKING.iterdir() if p.name != "codapath")
total_mb = sum(
    f.stat().st_size for p in remaining for f in ([p] if p.is_file() else p.rglob("*"))
    if f.is_file()
) / 1e6
print(f"\n/kaggle/working now holds {total_mb:.1f} MB (Output quota ~20 GB):")
for path in remaining:
    print(f"    {path.name}{'/' if path.is_dir() else ''}")

print(f"""
NEXT STEPS (no terminal needed)
  1. Output tab (right panel) -> download {ARCHIVE.name}.zip
  2. kaggle.com/datasets -> New Dataset -> upload that zip
     Kaggle extracts it into a directory named after the zip, so the cache
     files end up one level down. That is expected: run_al_baseline.ipynb and
     run_al_main.ipynb both search a few levels with `dir_containing`.
  3. In run_al_baseline.ipynb or run_al_main.ipynb: Add Data -> your new
     dataset. FEATURE_DIR is then resolved by filename, so there is no path
     to edit.""")